# Compare All Recommenders on Common Metrics

This notebook compares all exported recommender outputs on a shared test split using these metrics:

- Accuracy@K
- NDCG@K
- Diversity
- Personalisation
- Coverage

It auto-discovers every model file in `../bandits/saved_arms/*.csv` with columns: `UserID, rank, WineID`.

In [34]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f"{x:.6f}")

In [35]:
# Paths and config
PROJECT_ROOT = Path("..").resolve()
ARMS_DIR = PROJECT_ROOT / "bandits" / "saved_arms"
SPLIT_DIR = PROJECT_ROOT / "data" / "results" / "shared_split"
WINE_FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "wines_features.csv"
EXPORT_DIR = PROJECT_ROOT / "comparison"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

KS = [10]
PERSONALISATION_SAMPLE_PAIRS = 20_000
RANDOM_STATE = 42

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARMS_DIR:", ARMS_DIR)
print("SPLIT_DIR:", SPLIT_DIR)
print("WINE_FEATURES_PATH:", WINE_FEATURES_PATH)
print("EXPORT_DIR:", EXPORT_DIR)

PROJECT_ROOT: /Users/sofiiaavetisian/Desktop/UNI/recsys/FINAL_WINE/wine-recommendation-system
ARMS_DIR: /Users/sofiiaavetisian/Desktop/UNI/recsys/FINAL_WINE/wine-recommendation-system/bandits/saved_arms
SPLIT_DIR: /Users/sofiiaavetisian/Desktop/UNI/recsys/FINAL_WINE/wine-recommendation-system/data/results/shared_split
WINE_FEATURES_PATH: /Users/sofiiaavetisian/Desktop/UNI/recsys/FINAL_WINE/wine-recommendation-system/data/processed/wines_features.csv
EXPORT_DIR: /Users/sofiiaavetisian/Desktop/UNI/recsys/FINAL_WINE/wine-recommendation-system/comparison


In [36]:
# Load shared relevance labels (ground truth)
test_relevant_df = pd.read_csv(SPLIT_DIR / "test_relevant.csv")

def parse_relevant(value):
    if pd.isna(value):
        return set()
    s = str(value).strip()
    if not s:
        return set()
    return {int(tok) for tok in s.split() if tok}

relevant_by_user = {
    int(row.UserID): parse_relevant(row.RelevantWineIDs)
    for row in test_relevant_df.itertuples(index=False)
}

test_users = set(relevant_by_user.keys())
print("Users in shared test split:", len(test_users))

Users in shared test split: 5000


In [37]:
# Load item metadata used for diversity and coverage
wines_features = pd.read_csv(
    WINE_FEATURES_PATH,
    usecols=[
        "WineID", "Type_tok", "Elaborate_tok", "Body_tok",
        "Acidity_tok", "Country_tok", "RegionName_tok", "ABV_bin"
    ]
)

for col in wines_features.columns:
    if col != "WineID":
        wines_features[col] = wines_features[col].fillna("")

catalog_size = wines_features["WineID"].nunique()

item_signature = {
    int(r.WineID): (
        r.Type_tok, r.Elaborate_tok, r.Body_tok,
        r.Acidity_tok, r.Country_tok, r.RegionName_tok, r.ABV_bin
    )
    for r in wines_features.itertuples(index=False)
}

print("Catalog size:", catalog_size)
print("Signature dimensions:", len(next(iter(item_signature.values()))))

Catalog size: 100646
Signature dimensions: 7


In [38]:
# Metric helpers
def accuracy_at_k(relevant_set, recs, k):
    recs_k = recs[:k]
    if k == 0:
        return np.nan
    hits = sum(1 for i in recs_k if i in relevant_set)
    return hits / k


def ndcg_at_k(relevant_set, recs, k):
    recs_k = recs[:k]
    if not relevant_set:
        return np.nan

    dcg = 0.0
    for rank, item in enumerate(recs_k, start=1):
        if item in relevant_set:
            dcg += 1.0 / np.log2(rank + 1)

    ideal_hits = min(len(relevant_set), k)
    if ideal_hits == 0:
        return np.nan
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg


def intra_list_diversity(recs):
    # Average pairwise dissimilarity over metadata signatures.
    n = len(recs)
    if n < 2:
        return np.nan

    total_dissim = 0.0
    pairs = 0
    for i, j in combinations(range(n), 2):
        sig_i = item_signature.get(recs[i])
        sig_j = item_signature.get(recs[j])
        if sig_i is None or sig_j is None:
            continue

        equal_attrs = sum(1 for a, b in zip(sig_i, sig_j) if a == b)
        dissim = 1.0 - (equal_attrs / len(sig_i))
        total_dissim += dissim
        pairs += 1

    return total_dissim / pairs if pairs else np.nan


def personalisation_score(user_to_recs, max_pairs=20_000, random_state=42):
    # 1 - average Jaccard similarity between users' top-K lists.
    users = list(user_to_recs.keys())
    n = len(users)
    if n < 2:
        return np.nan

    rng = np.random.default_rng(random_state)
    rec_sets = {u: set(user_to_recs[u]) for u in users}

    all_pairs = n * (n - 1) // 2
    if all_pairs <= max_pairs:
        pairs = list(combinations(users, 2))
    else:
        pairs = []
        while len(pairs) < max_pairs:
            a, b = rng.choice(users, size=2, replace=False)
            if a > b:
                a, b = b, a
            pairs.append((a, b))

    jaccard_sims = []
    for a, b in pairs:
        sa, sb = rec_sets[a], rec_sets[b]
        union = sa | sb
        sim = (len(sa & sb) / len(union)) if union else 0.0
        jaccard_sims.append(sim)

    return 1.0 - float(np.mean(jaccard_sims))

In [39]:
# Evaluate one model file across all K values
def evaluate_model_file(csv_path, ks=(10)):
    recs = pd.read_csv(csv_path)
    required_cols = {"UserID", "rank", "WineID"}
    missing = required_cols - set(recs.columns)
    if missing:
        raise ValueError(f"{csv_path.name} missing required columns: {missing}")

    recs = recs.sort_values(["UserID", "rank"])

    model_name = csv_path.stem
    model_users = set(recs["UserID"].unique())
    common_users = sorted(model_users & test_users)

    rows = []
    for k in ks:
        topk_by_user = (
            recs[recs["rank"] <= k]
            .groupby("UserID")["WineID"]
            .apply(list)
            .to_dict()
        )

        eval_users = [u for u in common_users if u in topk_by_user]
        if not eval_users:
            continue

        acc_vals = []
        ndcg_vals = []
        div_vals = []
        user_rec_lists = {}
        all_recommended = set()

        for user_id in eval_users:
            rec_list = [int(x) for x in topk_by_user[user_id][:k]]
            user_rec_lists[user_id] = rec_list
            all_recommended.update(rec_list)

            relevant = relevant_by_user[user_id]
            acc_vals.append(accuracy_at_k(relevant, rec_list, k))
            ndcg_vals.append(ndcg_at_k(relevant, rec_list, k))
            div_vals.append(intra_list_diversity(rec_list))

        rows.append({
            "Model": model_name,
            "K": k,
            "EvalUsers": len(eval_users),
            "UserCoverageInTest": len(eval_users) / len(test_users),
            "Accuracy@K": float(np.nanmean(acc_vals)),
            "NDCG@K": float(np.nanmean(ndcg_vals)),
            "Diversity": float(np.nanmean(div_vals)),
            "Personalisation": personalisation_score(
                user_rec_lists,
                max_pairs=PERSONALISATION_SAMPLE_PAIRS,
                random_state=RANDOM_STATE,
            ),
            "Coverage": len(all_recommended) / catalog_size,
            "UniqueRecommendedItems": len(all_recommended),
            "CatalogSize": catalog_size,
        })

    return pd.DataFrame(rows)


In [40]:
# Run evaluation for all exported recommenders
arm_files = sorted(ARMS_DIR.glob("*.csv"))
if not arm_files:
    raise FileNotFoundError(f"No model files found in {ARMS_DIR}")

print("Discovered models:")
for p in arm_files:
    print("-", p.name)

all_results = []
for path in arm_files:
    model_df = evaluate_model_file(path, ks=KS)
    if model_df.empty:
        print(f"Skipping {path.name}: no evaluable users")
        continue
    all_results.append(model_df)

comparison_df = pd.concat(all_results, ignore_index=True)
comparison_df = comparison_df.sort_values(["K", "Accuracy@K"], ascending=[True, False])
comparison_df

Discovered models:
- als_recs.csv
- content_bert_recs.csv
- content_bow_recs.csv
- content_lemma_tfidf_recs.csv
- content_ner_tfidf_recs.csv
- content_tfidf_recs.csv
- context_als_recs.csv
- hybrid_tfidf_als_recs.csv
- popular_global_recs.csv
- random_normal_recs.csv


,Model,K,EvalUsers,UserCoverageInTest,Accuracy@K,NDCG@K,Diversity,Personalisation,Coverage,UniqueRecommendedItems,CatalogSize
7,hybrid_tfidf_als_recs,10,5000,1.000000,0.006620,0.016416,0.434351,0.996144,0.135773,13665,100646
0,als_recs,10,5000,1.000000,0.005160,0.010533,0.529584,0.993731,0.072472,7294,100646
6,context_als_recs,10,5000,1.000000,0.005160,0.010533,0.529584,0.993731,0.072472,7294,100646
5,content_tfidf_recs,10,5000,1.000000,0.004080,0.010596,0.223554,0.997045,0.178427,17958,100646
8,popular_global_recs,10,5000,1.000000,0.004000,0.008424,0.637199,0.035659,0.000169,17,100646
2,content_bow_recs,10,5000,1.000000,0.002000,0.005370,0.077930,0.998705,0.206774,20811,100646
3,content_lemma_tfidf_recs,10,5000,1.000000,0.001760,0.005103,0.146652,0.997466,0.184230,18542,100646
4,content_ner_tfidf_recs,10,5000,1.000000,0.000700,0.001744,0.149943,0.996212,0.236433,23796,100646
1,content_bert_recs,10,5000,1.000000,0.000560,0.001144,0.278714,0.988624,0.091668,9226,100646
9,random_normal_recs,10,5000,1.000000,0.000020,0.000026,0.641045,0.998470,0.061542,6194,100646


In [43]:
# Show models with partial user coverage (important for fair comparison)
coverage_flags = comparison_view[["Model", "K", "EvalUsers", "UserCoverageInTest"]].copy()
coverage_flags = coverage_flags.sort_values(["UserCoverageInTest", "Model", "K"], ascending=[True, True, True])
coverage_flags

,Model,K,EvalUsers,UserCoverageInTest
0,als_recs,10,5000,1.000000
1,content_bert_recs,10,5000,1.000000
2,content_bow_recs,10,5000,1.000000
3,content_lemma_tfidf_recs,10,5000,1.000000
4,content_ner_tfidf_recs,10,5000,1.000000
5,content_tfidf_recs,10,5000,1.000000
6,context_als_recs,10,5000,1.000000
7,hybrid_tfidf_als_recs,10,5000,1.000000
8,popular_global_recs,10,5000,1.000000
9,random_normal_recs,10,5000,1.000000


## Conclusion

Looking at the results, we can see that the overall performance of all models is relatively low in terms of accuracy and NDCG, which suggests that the recommendation task is quite challenging. However, there are still some clear differences between the models. The best performing model is the hybrid TF-IDF + ALS approach, which achieves the highest accuracy and NDCG. This indicates that combining collaborative filtering with content-based features helps improve the quality of recommendations, as it is able to capture both user preferences and item characteristics.

Similarly, the ALS-based models, including the standard ALS and context-aware ALS, also perform relatively well, although slightly worse than the hybrid model. This shows that collaborative filtering is effective in capturing user behavior patterns. On the other hand, the content-based models generally perform worse in terms of accuracy. For example, TF-IDF performs moderately, while more complex approaches such as BERT and NER-based models perform quite poorly. This could suggest that, in this dataset, textual features alone are not sufficient to accurately model user preferences.

At the same time, when we look at diversity and coverage, a different pattern appears. Content-based models tend to have much higher coverage and recommend a larger number of unique items. This means they explore the catalog more broadly, even though their recommendations are less accurate. In contrast, the popular global model has very low coverage and recommends only a small set of items, which explains why its personalisation score is also very low. It essentially gives almost the same recommendations to all users.

Another important observation is related to the personalisation metric. Many models, especially content-based ones and the random model, have very high personalisation scores. However, this is a false friend, because it does not mean that the recommendations are actually good or relevant. Instead, it only indicates that users receive different items, regardless of their quality. This is clearly visible in the random model, which has almost zero accuracy but very high personalisation.

Overall, the results show a clear trade-off. Models that focus on accuracy, such as ALS and the hybrid approach, tend to recommend fewer items and have lower coverage, while models that explore more, such as content-based approaches, achieve higher diversity and coverage but lower accuracy. Therefore, the hybrid model appears to provide the best balance between these aspects, making it the most effective approach among those tested.